# Regression trees
The study-hours examples, the split search by squared error, and a regression tree on the Boston housing data.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text

## Hours the day before the exam: a line fails, a tree works
The study-hours data are synthetic (fixed seeds), built to show the pattern.

In [ ]:
day = pd.read_csv("data/exam_day.csv")
line = LinearRegression().fit(day[["hours"]], day["marks"])
print("line R2 on the training data:", round(line.score(day[["hours"]], day["marks"]), 3))
print("line predicts for 5 hours:", round(line.predict(pd.DataFrame({"hours": [5]}))[0], 1))

# A tree with at most 3 leaves finds the two cuts by itself
tree = DecisionTreeRegressor(max_leaf_nodes=3, random_state=0).fit(day[["hours"]], day["marks"])
print(export_text(tree, feature_names=["hours"]))
print("tree R2:", round(tree.score(day[["hours"]], day["marks"]), 3))

In [ ]:
# The hand-made tree: means of the three groups (hours < 3, 3 to 6, 6 and more)
for lo, hi in [(0, 3), (3, 6), (6, 99)]:
    group = day["marks"][(day["hours"] >= lo) & (day["hours"] < hi)]
    print(f"{lo} <= hours < {hi}: {len(group)} students, mean {group.mean():.1f}")

## Scoring one split: the sum of squared errors

In [ ]:
hours = np.array([1, 2, 4, 5, 8])
marks = np.array([40, 46, 88, 90, 58])

def sse_of_split(t):
    # Each side predicts its own mean; add up the squared errors of both sides
    left, right = marks[hours <= t], marks[hours > t]
    return ((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum()

for t in [1.5, 3, 4.5, 6.5]:
    print(f"hours <= {t}: SSE = {sse_of_split(t):.1f}")

In [ ]:
# The same search over all 30 students: every midpoint between neighbouring hours
h, m = day["hours"].to_numpy(), day["marks"].to_numpy()
xs = np.unique(h)
cands = (xs[:-1] + xs[1:]) / 2
sse = [((m[h <= t] - m[h <= t].mean()) ** 2).sum() + ((m[h > t] - m[h > t].mean()) ** 2).sum() for t in cands]
print("best threshold:", cands[int(np.argmin(sse))], "SSE", round(min(sse), 1), "| SSE with no split:", round(((m - m.mean()) ** 2).sum(), 1))

## Two inputs: hours and CGPA

In [ ]:
two = pd.read_csv("data/exam_cgpa.csv")
t2 = DecisionTreeRegressor(max_depth=3, min_samples_leaf=4, random_state=0).fit(two[["hours", "cgpa"]], two["marks"])
print(export_text(t2, feature_names=["hours", "cgpa"]))

## Regression tree on the Boston housing data

In [ ]:
# load_boston was removed from scikit-learn in version 1.2; data/boston.csv is the same table (OpenML id 531)
boston = pd.read_csv("data/boston.csv")
X, y = boston.drop(columns="MEDV"), boston["MEDV"]       # MEDV: median house value, in thousands of dollars
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg = DecisionTreeRegressor(criterion="squared_error", max_depth=5, random_state=42)
reg.fit(X_train, y_train)
print("test R2:", round(r2_score(y_test, reg.predict(X_test)), 3))
print("5-fold CV R2 on the training set:", round(cross_val_score(reg, X_train, y_train, cv=5).mean(), 3))

In [ ]:
# Tune several hyperparameters at once with cross-validation
params = {"max_depth": [2, 4, 6, 8, None],
          "criterion": ["squared_error", "absolute_error"],
          "max_features": [0.25, 0.5, 1.0],
          "min_samples_split": [2, 0.05, 0.25]}
grid = GridSearchCV(DecisionTreeRegressor(random_state=42), params, cv=5)
grid.fit(X_train, y_train)
print(len(grid.cv_results_["params"]), "settings tried")
print("best:", grid.best_params_, "CV R2", round(grid.best_score_, 3), "test R2", round(grid.score(X_test, y_test), 3))

In [ ]:
# RandomizedSearchCV tries only n_iter random settings from the same grid: faster on big grids
rand = RandomizedSearchCV(DecisionTreeRegressor(random_state=42), params, n_iter=20, cv=5, random_state=42)
rand.fit(X_train, y_train)
print("best:", rand.best_params_, "CV R2", round(rand.best_score_, 3))

In [ ]:
# Feature importance: each column's share of the total impurity reduction in the tree (here: absolute error)
imp = pd.Series(grid.best_estimator_.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.round(3))
print("sum:", imp.sum())

In [ ]:
# The criterion values DecisionTreeRegressor accepts in this version
import sklearn
print(sklearn.__version__, [c.options for c in DecisionTreeRegressor._parameter_constraints["criterion"] if hasattr(c, "options")])